- Utilizando el código "sniffer" que es un programa que muestra el contenido del tráfico que llega, modificarlo para explicar cada paso que va ejecutando en pantalla.
```c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <sys/socket.h>
#include <arpa/inet.h>
#include <netinet/ip.h>
#include <netinet/ip_icmp.h>

int main() {
    int sockfd;
    char buffer[65536];
    struct sockaddr saddr;
    socklen_t saddr_len = sizeof(saddr);

    // Crear raw socket
    printf("Crea el socket por el que va a escuchar, usando tipo RAW y de protocolo ICMP \n");
    sockfd = socket(AF_INET, SOCK_RAW, IPPROTO_ICMP);
    if (sockfd < 0) {
        perror("Socket error");
        return 1;
    }
     
    printf("Sniffer ICMP iniciado... (Ctrl+C para detener)\n");

    while (1) {
         printf("Espera a recibir un paquete que llegue a traves del socket\n");
        ssize_t packet_size = recvfrom(sockfd, buffer, sizeof(buffer), 0, &saddr, &saddr_len);
        if (packet_size < 0) {
            perror("recvfrom error");
            continue;
        }
         printf("En el momento en el que llega, empieza a desarmar el paquete. \nSetea el inicio del header ip en la misma direccion del inicio del paquete. Como el struct del ip header tiene el tamano del header (valga la redundancia), va a apuntar al inicio de la cabezera terminando en su fin.");
        struct iphdr *ip = (struct iphdr*) buffer;
        if (ip->protocol == IPPROTO_ICMP) {
            printf("\n Hace lo mismo con icmphdr. Inicia la direccion del struct del icmp hdr donde termina el hdr del ip.");
            struct icmphdr *icmp = (struct icmphdr*)(buffer + ip->ihl * 4);
            printf("ICMP capturado: Tipo=%d Código=%d ID=%d Secuencia=%d\n",
                icmp->type,
                icmp->code,
                ntohs(icmp->un.echo.id),
                ntohs(icmp->un.echo.sequence)
            );

            // Guardar en archivo para analizar con Python
            FILE *f = fopen("icmp_packet.bin", "wb");
            if (f) {
                fwrite(buffer, 1, packet_size, f);
                fclose(f);
                printf("Paquete guardado en icmp_packet.bin\n");
                break; // solo capturamos uno para el ejemplo
            }
        }
    }

    close(sockfd);
    return 0;
}
```
- Modificar el sniffer para que vaya mostrando todo el tráfico que llega.
```c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <sys/socket.h>
#include <arpa/inet.h>
#include <netinet/ip.h>
#include <netinet/ip_icmp.h>

int main() {
    int sockfd;
    char buffer[65536];
    struct sockaddr saddr;
    socklen_t saddr_len = sizeof(saddr);

    // Crear raw socket
    sockfd = socket(AF_INET, SOCK_RAW, IPPROTO_ICMP);
    if (sockfd < 0) {
        perror("Socket error");
        return 1;
    }

    printf("Sniffer ICMP iniciado... (Ctrl+C para detener)\n");

    while (1) {
        ssize_t packet_size = recvfrom(sockfd, buffer, sizeof(buffer), 0, &saddr, &saddr_len);
        if (packet_size < 0) {
            perror("recvfrom error");
            continue;
        }

        struct iphdr *ip = (struct iphdr*) buffer;
        if (ip->protocol == IPPROTO_ICMP) {
            struct icmphdr *icmp = (struct icmphdr*)(buffer + ip->ihl * 4);
            printf("ICMP capturado: Tipo=%d Código=%d ID=%d Secuencia=%d\n",
                icmp->type,
                icmp->code,
                ntohs(icmp->un.echo.id),
                ntohs(icmp->un.echo.sequence)
            );
        }
    }

    close(sockfd);
    return 0;
}
```
- Enviar tráfico al "sniffer" desde el cliente escrito en la parte A del TP1
```c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <sys/socket.h>
#include <arpa/inet.h>
#include <netinet/ip.h>
#include <netinet/ip_icmp.h>
#include <netinet/tcp.h>
int main() {
    int sockfd;
    char buffer[65536];
    struct sockaddr saddr;
    socklen_t saddr_len = sizeof(saddr);

    // Crear raw socket
    sockfd = socket(AF_INET, SOCK_RAW, IPPROTO_TCP);
    if (sockfd < 0) {
        perror("Socket error");
        return 1;
    }

    printf("Sniffer ICMP iniciado... (Ctrl+C para detener)\n");

    while (1) {
        ssize_t packet_size = recvfrom(sockfd, buffer, sizeof(buffer), 0, &saddr, &saddr_len);
        if (packet_size < 0) {
            perror("recvfrom error");
            continue;
        }
        struct iphdr *ip = (struct iphdr*) buffer;
        if (ip->protocol == IPPROTO_TCP ) {
            struct tcphdr *tcp = (struct tcphdr*)(buffer + ip->ihl * 4);
            if(ntohs(tcp->source) ==8080){
            printf("TCP capturado: Secuencia=%d, Ventana tamano =%d Destino:%d. ; Mensaje:%s \n",
                tcp->seq,
                tcp->window,
                ntohs(tcp->dest),
                ((unsigned char *)(tcp)+(tcp->doff)*4));}
            /*Aca imprime basura pero no por este codigo sino por el cliente
            que no se asegura de que los strings terminen con \n*/
        }
    }

    close(sockfd);
    return 0;
}
```
- Enviar tráfico ICMP al "sniffer" y mostrar los resultados con un LOG con comentarios.
```c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <sys/socket.h>
#include <arpa/inet.h>
#include <netinet/ip.h>
#include <netinet/ip_icmp.h>
#include <time.h>

#define LOG_FILE "icmp_sniffer.log"

void escribir_log(const char *mensaje) {
    FILE *log = fopen(LOG_FILE, "a");
    if (!log) {
        perror("Error al abrir el archivo de log");
        return;
    }

    time_t ahora = time(NULL);
    struct tm *t = localtime(&ahora);

    fprintf(log, "%04d-%02d-%02d %02d:%02d:%02d - %s\n",
            t->tm_year + 1900, t->tm_mon + 1, t->tm_mday,
            t->tm_hour, t->tm_min, t->tm_sec,
            mensaje);

    fclose(log);
}

int main() {
    int sockfd;
    char buffer[65536];
    struct sockaddr saddr;
    socklen_t saddr_len = sizeof(saddr);

    // Crear raw socket
    sockfd = socket(AF_INET, SOCK_RAW, IPPROTO_ICMP);
    if (sockfd < 0) {
        perror("Socket error");
        return 1;
    }

    printf("Sniffer ICMP iniciado... (Ctrl+C para detener)\n");
    escribir_log("Sniffer ICMP iniciado");

    while (1) {
        ssize_t packet_size = recvfrom(sockfd, buffer, sizeof(buffer), 0,
                                       &saddr, &saddr_len);
        if (packet_size < 0) {
            perror("recvfrom error");
            escribir_log("Error en recvfrom()");
            continue;
        }

        struct iphdr *ip = (struct iphdr *)buffer;

        if (ip->protocol == IPPROTO_ICMP) {
            struct icmphdr *icmp =
                (struct icmphdr *)(buffer + ip->ihl * 4);

            struct sockaddr_in *addr_in =
                (struct sockaddr_in *)&saddr;

            char src_ip[INET_ADDRSTRLEN];
            inet_ntop(AF_INET, &(addr_in->sin_addr),
                      src_ip, INET_ADDRSTRLEN);

            char log_msg[256];
            snprintf(log_msg, sizeof(log_msg),
                     "ICMP capturado: Origen=%s Tipo=%d Código=%d ID=%d Secuencia=%d Tamaño=%ld bytes",
                     src_ip,
                     icmp->type,
                     icmp->code,
                     ntohs(icmp->un.echo.id),
                     ntohs(icmp->un.echo.sequence),
                     packet_size);

            
            printf("%s\n", log_msg);

            escribir_log(log_msg);
        }
    }

    close(sockfd);
    return 0;
}

- Explicar las estructuras del código de ICMP y del Sniffer, especialmente los campos de protocolo.
- Estudiar el "sniffer" para que muestre IP, TCP y UDP.
- Mostrar resultados.

In [ ]:
```c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <sys/socket.h>
#include <arpa/inet.h>
#include <netinet/ip.h>
#include <netinet/ip_icmp.h>
#include <netinet/tcp.h>
#include <netinet/udp.h>
#include <linux/if_packet.h>
#include <linux/if_ether.h>

int main() {
    int sockfd;
    char buffer[65536];
    struct sockaddr saddr;
    socklen_t saddr_len = sizeof(saddr);


    sockfd = socket(AF_PACKET, SOCK_RAW, htons(ETH_P_IP));
    if (sockfd < 0) {
        perror("Socket error");
        return 1;
    }

    printf("Sniffer iniciado... (Ctrl+C para detener)\n");

    while (1) {
        ssize_t packet_size = recvfrom(sockfd, buffer, sizeof(buffer), 0, &saddr, &saddr_len);
        if (packet_size < 0) {
            perror("recvfrom error");
            continue;
        }

        struct iphdr *ip = (struct iphdr*)(buffer + sizeof(struct ethhdr));
        printf("IP %s -> %s | VER=%d IHL=%d TTL=%d PROTO=%d LEN=%d ID=%d\n",
           inet_ntoa(*(struct in_addr *)&ip->saddr),
           inet_ntoa(*(struct in_addr *)&ip->daddr),
           ip->version,
           ip->ihl * 4,
           ip->ttl,
           ip->protocol,
           ntohs(ip->tot_len),
           ntohs(ip->id));
        switch(ip->protocol){
            case(IPPROTO_ICMP): {
                struct icmphdr *icmp = (struct icmphdr*)((unsigned char*)ip + ip->ihl * 4);
                printf("ICMP capturado: Tipo=%d Codigo: %d ID=%d SEQ=%d \n",
                    icmp->type,
                    icmp->code,
                    ntohs(icmp->un.echo.id),
                    ntohs(icmp->un.echo.sequence)
                );
                break;
            }
            case(IPPROTO_TCP):{
               struct tcphdr *tcp = (struct tcphdr*)((unsigned char*)ip + ip->ihl * 4);
               printf("TCP %s:%u -> %s:%u | SEQ=%u ACK=%u |WIN=%u LEN=%u\n",
                   inet_ntoa(*(struct in_addr *)&ip->saddr),
                   ntohs(tcp->source),
                   inet_ntoa(*(struct in_addr *)&ip->daddr),
                   ntohs(tcp->dest),
                   ntohl(tcp->seq),
                   ntohl(tcp->ack_seq),
                   ntohs(tcp->window),
                   ntohs(ip->tot_len) - (ip->ihl * 4) - (tcp->doff * 4));
                break;
                }
            case(IPPROTO_UDP):{
                struct udphdr *udp = (struct udphdr*)((unsigned char*)ip + ip->ihl * 4);
                  printf("UDP %s:%u -> %s:%u | LEN=%u\n",
                       inet_ntoa(*(struct in_addr *)&ip->saddr),ntohs(udp->source),
                       inet_ntoa(*(struct in_addr *)&ip->daddr), ntohs(udp->dest),
                       ntohs(udp->len) - sizeof(struct udphdr));
                                                       
                break;
            }
        }
    }

    close(sockfd);
    return 0;
}
```

#### Para raw_sockets_tcp.c
- El programa anterior arma paquete IP + TCP.
- Lo envía como un SYN (inicio de conexión TCP).
- Calcula los checksums (IP y TCP).
- Requiere IP de origen válida.
- Probar con IPs distintas de localhost
- Verlo con tcpdump
- Mientras se ejecuta el programa, en otra terminal ejecutar tcpdump (reemplazar por interfaz correcta si es necesario)
  


-salida generada en tcpdump luego de cambiar la ip a una inventada 1.2.3.9 y el puerto 9999.
-18:53:10.352990 IP 1.2.3.9.9999 > localhost.http: Flags [S], seq 0, win 5840, length 0